# Fixed Effects Analysis — Twitter Sentiment & Stock Returns
**Ashley Thompson — Capstone**

Models 1–4 + robustness checks + publication-ready LaTeX table export.

> No GPU needed — standard Colab CPU runtime is fine for this notebook.

## 1. Install dependencies

In [ ]:
!pip install -q pyfixest

## 2. Load data
**Option A — Google Drive** | **Option B — Manual upload**  
Run one, skip the other.

In [ ]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

In [ ]:
# Option B — Manual upload (data only)
from google.colab import files
uploaded = files.upload()
DATA_PATH = 'panel_long.csv'

In [ ]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Setup

In [ ]:
import numpy as np
import pandas as pd
import pyfixest as pf
import os

np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')
long.head()

## 4. Model 1 — Simple FE
Baseline within-firm estimate of Twitter sentiment on same-day return.

In [ ]:
m1 = pf.feols('return ~ twitter_sent | ticker', data=long, vcov='hetero')
m1.summary()

## 5. Model 2 — FE with confounder matrix
Adds fundamentals and technical indicators. Excludes `px_open` and `px_close` (collinear with return).

In [ ]:
confounders = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'news_sent', 'rsi_30', 'ma_50', 'twitter_neg_count',
]
m2 = pf.feols(
    f"return ~ twitter_sent + {' + '.join(confounders)} | ticker",
    data=long, vcov='hetero'
)
m2.summary()

## 6. Model 3 — Continuous negative sentiment treatment
`neg_twitter_sent = clip(twitter_sent, max=0)`: zero on positive days, raw score on negative days.

In [ ]:
long['neg_twitter_sent'] = long['twitter_sent'].clip(upper=0)
m3 = pf.feols('return ~ neg_twitter_sent | ticker', data=long, vcov='hetero')
m3.summary()

## 7. Model 4 — Negative tweet count as standalone treatment
Per Teti et al. (2019): polarity-broken counts outperform total count. `twitter_neg_count` sidesteps neutral-tweet dilution in the Bloomberg index.

In [ ]:
m4 = pf.feols('return ~ twitter_neg_count | ticker', data=long, vcov='hetero')
m4.summary()

## 8. Placebo test
Does today's `twitter_sent` predict *yesterday's* return (`lag1`)? Coefficient should be insignificant under a causal story.

In [ ]:
placebo = pf.feols('lag1 ~ twitter_sent | ticker', data=long, vcov='hetero')
placebo.summary()

## 9. No-reversal test — Gu & Kurov (2020)
All lagged sentiment variables estimated jointly. Only `sent_lag1` should be significant under a causal information story.

In [ ]:
for n in [1, 2, 3, 5, 7]:
    long[f'sent_lag{n}'] = long.groupby('ticker')['twitter_sent'].shift(n)

no_reversal = pf.feols(
    'return ~ sent_lag1 + sent_lag2 + sent_lag3 + sent_lag5 + sent_lag7 | ticker',
    data=long, vcov='hetero'
)
no_reversal.summary()

## 10. Export — Publication-ready LaTeX tables
Saves two `.tex` files to your output folder:
- `fe_main_results.tex` — Models 1–4 side by side
- `fe_robustness.tex` — Placebo and no-reversal test

In [ ]:
# Main results table
main_tex = pf.etable(
    [m1, m2, m3, m4],
    type       = 'tex',
    signif_code = {0.01: '***', 0.05: '**', 0.1: '*'},
    notes      = 'Heteroskedasticity-robust SE in parentheses. Company fixed effects in all models. Model 3 treatment is clip(twitter\_sent, max=0). Model 4 per Teti et al. (2019).'
)
with open(OUTPUT_PATH + 'fe_main_results.tex', 'w') as f:
    f.write(main_tex)
print('Saved fe_main_results.tex')
print(main_tex)

In [ ]:
# Robustness table
robustness_tex = pf.etable(
    [placebo, no_reversal],
    type        = 'tex',
    signif_code = {0.01: '***', 0.05: '**', 0.1: '*'},
    notes       = 'Placebo: lag1 $\\sim$ twitter\_sent $|$ ticker. No-reversal: all sentiment lags estimated jointly. Only sent\_lag1 should be significant under a causal interpretation (Gu \\& Kurov, 2020).'
)
with open(OUTPUT_PATH + 'fe_robustness.tex', 'w') as f:
    f.write(robustness_tex)
print('Saved fe_robustness.tex')
print(robustness_tex)

In [ ]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])

staged   = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(
    ['git', 'log', 'origin/main..HEAD', '--oneline'],
    capture_output=True, text=True
)

if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add FE analysis outputs from Colab'], check=True)

if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit or push.')